# Swin-V2-Tiny + CBAM-UNet + Deep Supervision

**Notebook:** `train_2_swin_v2_tiny.ipynb`  
**Output dir:** `outputs/swin_v2_tiny/`  
**Hardware:** Kaggle T4 — **single GPU + AMP** (DataParallel is broken with timm ConvNeXt/Swin under AMP — see notebook header for explanation)  
**Batch size:** 12  ×  grad_accum 1  =  effective 12  

## Rationale
Self-attention captures non-local feature similarity, which is exactly what copy-move forgery requires (matching distant patches).

## ⚠ If you previously ran this notebook with DataParallel
**Click `Run → Restart Kernel & Clear All Outputs`, then `Run All`.**
A stale `nn.DataParallel` wrapper in kernel memory will trigger a
`misaligned address` error even after the notebook is updated, because
the old wrapper persists across cell re-runs.

## Crash-safety defaults applied
- `num_workers=0` (eliminates `/dev/shm` worker-death failures — original crash root cause)
- DataParallel **permanently disabled** (incompatible with AMP + ConvNeXt/Swin)
- Defensive unwrap of any stale `DataParallel` wrap from prior runs
- Smoke-test cell hard-asserts no DP wrap before the first forward pass
- Modern AMP API with legacy fallback
- Smoke-test cell runs 1 fwd+bwd pass before the full loop
- Periodic `torch.cuda.empty_cache()` between train and val phases

## Reproducibility
- `SEED=42` and identical UID-level split as `review2.ipynb`  
- The same 713-sample validation set is used by every model for fair comparison.

## End-of-notebook contract
- Saves `outputs/swin_v2_tiny/model.pt` (state dict + threshold + meta)
- Saves `metrics.json`, `history.csv`, `dashboard.png`, `predictions.png`, `training_curves.png`
- Releases all CUDA memory and prints peak VRAM

## 1. Environment Setup

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'albumentations>=1.4,<2.0', 'timm>=0.9.16', 'seaborn'],
    check=True
)
print('Packages installed.')

## 2. Imports

In [ ]:
import os, sys, gc, json, random, time, warnings, csv
from pathlib import Path
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import (
    roc_auc_score, accuracy_score, confusion_matrix,
    f1_score, roc_curve, classification_report,
)
from scipy.ndimage import label as scipy_label
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# Modern AMP API (PyTorch 2.4+). Falls back to legacy if unavailable.
try:
    from torch.amp import GradScaler as _AmpGradScaler, autocast as _amp_autocast
    def make_scaler():
        return _AmpGradScaler('cuda')
    def autocast():
        return _amp_autocast('cuda')
except ImportError:
    from torch.cuda.amp import GradScaler as _AmpGradScaler, autocast as _legacy_autocast
    def make_scaler():
        return _AmpGradScaler()
    def autocast():
        return _legacy_autocast()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {DEVICE}')
print(f'Albu     : {A.__version__}')
print(f'Timm     : {timm.__version__}')
if torch.cuda.is_available():
    print(f'CUDA cnt : {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        cap   = torch.cuda.get_device_capability(i)
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        free, _ = torch.cuda.mem_get_info(i)
        print(f'  GPU {i} : {torch.cuda.get_device_name(i)} '
              f'(sm_{cap[0]}{cap[1]})  total={total:.1f} GB  free={free/1e9:.1f} GB')
    # /dev/shm size — shared memory used by DataLoader workers
    try:
        import shutil
        shm = shutil.disk_usage('/dev/shm')
        print(f'/dev/shm : total={shm.total/1e9:.2f} GB  free={shm.free/1e9:.2f} GB')
    except Exception:
        pass

## 3. Configuration

In [ ]:
# ── Model identity ───────────────────────────────────────────────────────────
MODEL_NAME = 'swin_v2_tiny'
DISPLAY    = 'Swin-V2-Tiny + CBAM-UNet + Deep Supervision'
TIMM_NAME  = 'swinv2_tiny_window8_256'

# ── Paths ────────────────────────────────────────────────────────────────────
INPUT_DIR   = '/kaggle/input/datasets/llkh0a/recod-ailuc-scientific-image-forgery-detection'
if not os.path.isdir(INPUT_DIR):
    INPUT_DIR = '/kaggle/input/recod-ailuc-scientific-image-forgery-detection'
TRAIN_IMGS  = f'{INPUT_DIR}/train_images'
TRAIN_MASKS = f'{INPUT_DIR}/train_masks'
SUPP_IMGS   = f'{INPUT_DIR}/supplemental_images'
SUPP_MASKS  = f'{INPUT_DIR}/supplemental_masks'

OUT_DIR = Path('/kaggle/working/outputs') / MODEL_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_PATH  = OUT_DIR / 'best.pt'
FINAL_PATH = OUT_DIR / 'model.pt'

# ── Training ─────────────────────────────────────────────────────────────────
IMG_SIZE   = 512
BATCH_SIZE = 12
GRAD_ACCUM = 1
MAX_EPOCHS = 60
LR         = 0.0003
ENC_LR     = LR * 0.1
PATIENCE   = 8
VAL_FRAC   = 0.15
SEED       = 42
CLIP_NORM  = 1.0
USE_GRAD_CHECKPOINT = False

# ── Loss weights ─────────────────────────────────────────────────────────────
CLS_WEIGHT = 2.0
SEG_WEIGHT = 1.0
AUX_WEIGHT = 0.4

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Reproducibility — Python/NumPy/Torch RNGs deterministic, cuDNN kernels NOT
# strictly deterministic (cudnn.deterministic=True breaks DataParallel + AMP
# + ConvNeXt/Swin permute flows with 'misaligned address' on cuBLAS GEMM).
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic    = False
    torch.backends.cudnn.benchmark        = True   # autotune for fixed shape
    torch.backends.cudnn.allow_tf32       = True
    torch.backends.cuda.matmul.allow_tf32 = True

print(f'Model    : {DISPLAY}')
print(f'Output   : {OUT_DIR}')
print(f'BS={BATCH_SIZE}  GradAcc={GRAD_ACCUM}  LR={LR:.0e}  Enc LR={ENC_LR:.0e}  Epochs={MAX_EPOCHS}')

## 4. Dataset & Splits

In [ ]:
def load_mask(path, h, w):
    """Load .npy ground-truth mask, union over multi-region, resize if needed."""
    m = np.load(path)
    if m.ndim == 3:
        m = m.max(axis=0)
    if m.shape != (h, w):
        m = cv2.resize(m.astype(np.float32), (w, h), interpolation=cv2.INTER_NEAREST)
    return (m > 0).astype(np.float32)


class ForgeryDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = cv2.cvtColor(cv2.imread(s['img']), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        mask = (load_mask(s['mask'], h, w)
                if s['label'] == 1 and s['mask']
                else np.zeros((h, w), dtype=np.float32))
        if self.transform:
            out  = self.transform(image=img, mask=mask)
            img  = out['image']
            mask = out['mask']
        return img, torch.tensor(s['label'], dtype=torch.float32), mask.unsqueeze(0)


def build_samples():
    samples, uids = [], []
    auth_dir = f'{TRAIN_IMGS}/authentic'
    forg_dir = f'{TRAIN_IMGS}/forged'
    auth_uids = {os.path.splitext(f)[0] for f in os.listdir(auth_dir) if f.endswith('.png')}
    forg_uids = {os.path.splitext(f)[0] for f in os.listdir(forg_dir) if f.endswith('.png')}
    for uid in sorted(auth_uids & forg_uids):
        uids.append(uid)
        samples.append({'uid': uid, 'img': f'{auth_dir}/{uid}.png',
                        'label': 0, 'mask': None})
        mp = f'{TRAIN_MASKS}/{uid}.npy'
        if os.path.exists(mp):
            samples.append({'uid': uid, 'img': f'{forg_dir}/{uid}.png',
                            'label': 1, 'mask': mp})
    if os.path.isdir(SUPP_IMGS):
        for fname in sorted(os.listdir(SUPP_IMGS)):
            if not fname.endswith('.png'):
                continue
            uid = os.path.splitext(fname)[0]
            mp  = f'{SUPP_MASKS}/{uid}.npy'
            if os.path.exists(mp):
                samples.append({'uid': uid, 'img': f'{SUPP_IMGS}/{fname}',
                                'label': 1, 'mask': mp, 'supplemental': True})
    return samples, uids


all_samples, all_uids = build_samples()
shuffled    = sorted(all_uids)
random.shuffle(shuffled)
n_val       = int(VAL_FRAC * len(shuffled))
val_uids    = set(shuffled[:n_val])
train_uids  = set(shuffled[n_val:])

train_samples = [s for s in all_samples if s.get('supplemental') or s['uid'] in train_uids]
val_samples   = [s for s in all_samples if s['uid'] in val_uids]

print(f'Train: {len(train_samples)}  '
      f'(auth={sum(s["label"]==0 for s in train_samples)}, '
      f'forg={sum(s["label"]==1 for s in train_samples)})')
print(f'Val  : {len(val_samples)}  '
      f'(auth={sum(s["label"]==0 for s in val_samples)}, '
      f'forg={sum(s["label"]==1 for s in val_samples)})')


## 5. Augmentation

In [ ]:
train_tf = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.5, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10),
        A.CLAHE(clip_limit=2.0),
    ], p=0.6),
    A.OneOf([
        A.ImageCompression(quality_range=(70, 100)),
        A.GaussianBlur(blur_limit=(3, 5)),
        A.GaussNoise(std_range=(0.02, 0.1)),
    ], p=0.4),
    A.CoarseDropout(num_holes_range=(1, 4),
                    hole_height_range=(16, 48),
                    hole_width_range=(16, 48),
                    fill=0, p=0.2),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])
print('Augmentation pipelines ready.')


## 6. DataLoaders

In [ ]:
train_ds = ForgeryDataset(train_samples, train_tf)
val_ds   = ForgeryDataset(val_samples,   val_tf)

train_labels   = [s['label'] for s in train_samples]
class_counts   = np.bincount(train_labels)
sample_weights = [1.0 / class_counts[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

# num_workers=0 on Kaggle: avoids /dev/shm pressure that silently kills workers
# and triggers main-loop hangs. Slightly slower I/O but rock-solid stability.
NUM_WORKERS = int(os.environ.get('NUM_WORKERS', '0'))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                          persistent_workers=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=False)
print(f'Train batches: {len(train_loader)}   Val batches: {len(val_loader)}'
      f'   workers={NUM_WORKERS}')


## 7. Model Architecture

In [ ]:
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.channel_fc = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
        )
        self.spatial_conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        b, c = x.shape[:2]
        avg = self.channel_fc(F.adaptive_avg_pool2d(x, 1).view(b, c)).view(b, c, 1, 1)
        mx  = self.channel_fc(F.adaptive_max_pool2d(x, 1).view(b, c)).view(b, c, 1, 1)
        x = x * torch.sigmoid(avg + mx)
        spatial = torch.cat([x.mean(1, keepdim=True), x.max(1, keepdim=True)[0]], dim=1)
        x = x * torch.sigmoid(self.spatial_conv(spatial))
        return x


class ConvBNReLU(nn.Sequential):
    def __init__(self, in_c, out_c, k=3, p=1):
        super().__init__(
            nn.Conv2d(in_c, out_c, k, padding=p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )


class DecoderBlock(nn.Module):
    def __init__(self, in_c, skip_c, out_c):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv = nn.Sequential(
            ConvBNReLU(in_c + skip_c, out_c),
            ConvBNReLU(out_c, out_c),
        )
        self.cbam = CBAM(out_c)

    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
        return self.cbam(self.conv(x))

class FourLevelUNet(nn.Module):
    """UNet for 4-level encoders (ConvNeXt / Swin).

    Uses the known expected channel count from feature_info to
    disambiguate NCHW vs NHWC layout robustly. timm's Swin-V2
    returns NHWC; ConvNeXt returns NCHW.
    """
    def __init__(self, encoder, ch, channels_last_input=False):
        super().__init__()
        self.encoder = encoder
        self.expected_channels = list(ch)
        self.channels_last_input = channels_last_input
        self.d3 = DecoderBlock(ch[3], ch[2], 256)
        self.d2 = DecoderBlock(256,   ch[1], 128)
        self.d1 = DecoderBlock(128,   ch[0],  64)
        self.d0a = DecoderBlock(64,        0,  32)
        self.d0b = DecoderBlock(32,        0,  16)
        self.seg_head  = nn.Conv2d(16, 1, 1)
        self.aux3_head = nn.Conv2d(256, 1, 1)
        self.aux2_head = nn.Conv2d(128, 1, 1)
        self.cls_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(ch[3], 512), nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 1),
        )

    @staticmethod
    def _ensure_nchw(t, expected_c):
        if t.dim() != 4:
            return t
        if t.shape[1] == expected_c:    # already NCHW
            return t
        if t.shape[-1] == expected_c:   # NHWC -> permute
            return t.permute(0, 3, 1, 2).contiguous()
        return t  # fall through; shape might be ambiguous

    def forward(self, x):
        feats = self.encoder(x)
        feats = [self._ensure_nchw(f, c)
                 for f, c in zip(feats, self.expected_channels)]
        f0, f1, f2, f3 = feats
        cls_out = self.cls_head(f3)
        d3 = self.d3(f3, f2)
        d2 = self.d2(d3, f1)
        d1 = self.d1(d2, f0)
        d0a = self.d0a(d1)
        d0  = self.d0b(d0a)
        seg = self.seg_head(d0)
        if seg.shape[-2:] != x.shape[-2:]:
            seg = F.interpolate(seg, size=x.shape[-2:], mode='bilinear', align_corners=False)
        return cls_out, seg, self.aux3_head(d3), self.aux2_head(d2)

# Build encoder + model
encoder = timm.create_model('swinv2_tiny_window8_256', pretrained=True, features_only=True, img_size=512)
channels = encoder.feature_info.channels()
print(f'Encoder feature channels: {channels}')

model = FourLevelUNet(encoder, channels, channels_last_input=True).to(DEVICE)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Params: total={total:,}  trainable={trainable:,}')


## 8. Loss Functions

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p   = torch.sigmoid(logits)
        pt  = targets * p + (1 - targets) * (1 - p)
        at  = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        return (at * (1 - pt) ** self.gamma * bce).mean()


def tversky_loss(logits, targets, alpha=0.3, beta=0.7, eps=1e-7):
    p  = torch.sigmoid(logits)
    tp = (p * targets).sum()
    fp = (p * (1 - targets)).sum()
    fn = ((1 - p) * targets).sum()
    return 1 - (tp + eps) / (tp + alpha * fp + beta * fn + eps)


def seg_loss(pred, gt):
    return F.binary_cross_entropy_with_logits(pred, gt) + tversky_loss(pred, gt)


_focal = FocalLoss(alpha=0.5, gamma=2.0)


def multitask_loss(cls_out, seg_out, aux_a, aux_b, labels, masks):
    cls_l = _focal(cls_out.squeeze(1), labels)
    seg_l = torch.tensor(0.0, device=cls_out.device)
    forged = labels == 1
    if forged.any():
        gt_a = F.adaptive_avg_pool2d(masks[forged], aux_a.shape[-2:])
        gt_b = F.adaptive_avg_pool2d(masks[forged], aux_b.shape[-2:])
        seg_l = (
            seg_loss(seg_out[forged], masks[forged])
            + AUX_WEIGHT * seg_loss(aux_a[forged], gt_a)
            + AUX_WEIGHT * seg_loss(aux_b[forged], gt_b)
        )
    total = CLS_WEIGHT * cls_l + SEG_WEIGHT * seg_l
    return total, cls_l.detach(), seg_l.detach()


print('Loss functions ready.')


## 9. Optimiser & Scheduler

In [ ]:
enc_params = list(model.encoder.parameters())
dec_params = [p for n, p in model.named_parameters() if not n.startswith('encoder')]

optimizer = torch.optim.AdamW(
    [
        {'params': enc_params, 'lr': ENC_LR},
        {'params': dec_params, 'lr': LR},
    ],
    weight_decay=1e-4,
)

warmup = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=5)
cosine = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS - 5, eta_min=LR * 0.01)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[5])

# DataParallel is permanently DISABLED. It's incompatible with AMP + timm
# ConvNeXt/Swin (cuBLAS misaligned-address bug in Linear-after-permute) and
# has been deprecated by PyTorch. Single-GPU FP16 on T4 is ~4x faster than
# 2-GPU FP32 anyway. To use multiple GPUs, switch to DDP via a .py script.
#
# Defensive unwrap: if a stale DataParallel-wrapped model lingers in kernel
# state (e.g. from a previous training attempt), strip the wrapper here.
if isinstance(model, nn.DataParallel):
    print('Stripping stale DataParallel wrapper from model.')
    model = model.module
if torch.cuda.device_count() > 1:
    print(f'{torch.cuda.device_count()} GPUs visible; using GPU 0 only '
          f'(DataParallel removed; single-GPU FP16 is ~4x faster on T4).')

USE_AMP = torch.cuda.is_available()
scaler  = make_scaler() if USE_AMP else None
print(f'AMP: {USE_AMP}  |  GradAccum: {GRAD_ACCUM}  |  Effective batch: {BATCH_SIZE * GRAD_ACCUM}')


## 10. Training & Validation Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, epoch):
    model.train()
    losses, cls_ls, seg_ls = [], [], []
    pbar = tqdm(loader, desc=f'Epoch {epoch:02d} [Train]', leave=False)
    accum = 0
    optimizer.zero_grad(set_to_none=True)
    for imgs, labels, masks in pbar:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        masks  = masks.to(DEVICE, non_blocking=True)

        if USE_AMP:
            with autocast():
                cls_out, seg_out, aux_a, aux_b = model(imgs)
                loss, cl, sl = multitask_loss(cls_out, seg_out, aux_a, aux_b, labels, masks)
                loss = loss / GRAD_ACCUM
        else:
            cls_out, seg_out, aux_a, aux_b = model(imgs)
            loss, cl, sl = multitask_loss(cls_out, seg_out, aux_a, aux_b, labels, masks)
            loss = loss / GRAD_ACCUM

        if not torch.isfinite(loss):
            optimizer.zero_grad(set_to_none=True)
            continue

        if USE_AMP:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        accum += 1
        if accum >= GRAD_ACCUM:
            if USE_AMP:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            accum = 0

        losses.append(loss.item() * GRAD_ACCUM)
        cls_ls.append(cl.item())
        seg_ls.append(sl.item())
        pbar.set_postfix({
            'loss': f'{np.mean(losses):.4f}',
            'cls':  f'{np.mean(cls_ls):.4f}',
            'seg':  f'{np.mean(seg_ls):.4f}',
        })

    return float(np.mean(losses)), float(np.mean(cls_ls)), float(np.mean(seg_ls))


@torch.no_grad()
def validate(model, loader, threshold=0.5):
    model.train(False)
    y_true, y_prob, dice_scores = [], [], []
    for imgs, labels, masks in tqdm(loader, desc='Val', leave=False):
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        masks  = masks.to(DEVICE, non_blocking=True)
        cls_out, seg_out, _, _ = model(imgs)
        probs = torch.sigmoid(cls_out).squeeze(1)
        y_true.extend(labels.cpu().numpy())
        y_prob.extend(probs.cpu().numpy())
        for i in range(len(labels)):
            if labels[i].item() == 1:
                pred  = (torch.sigmoid(seg_out[i, 0]) > threshold).float()
                gt    = masks[i, 0]
                inter = (pred * gt).sum()
                dice_scores.append((2 * inter / (pred.sum() + gt.sum() + 1e-7)).item())

    y_pred = (np.array(y_prob) > threshold).astype(int)
    acc    = accuracy_score(y_true, y_pred)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = 0.5
    dice  = float(np.mean(dice_scores)) if dice_scores else 0.0
    cm    = confusion_matrix(y_true, y_pred)
    fgrec = (cm[1, 1] / (cm[1, 0] + cm[1, 1] + 1e-7) if cm.shape == (2, 2) else 0.0)
    comp  = 0.4 * auc + 0.3 * fgrec + 0.3 * dice
    return {'acc': acc, 'auc': auc, 'dice': dice, 'fgrec': fgrec, 'comp': comp}


print('Training & validation functions ready.')


## 11a. cuDNN preset for DataParallel + AMP

In [ ]:
# ── DataParallel + AMP safety: cuDNN tuning ─────────────────────────────────
# Ensures the 'misaligned address' bug does not hit the cuBLAS GEMM inside
# ConvNeXt/Swin MLP blocks when DataParallel scatters a half-batch onto GPU 1.
# Idempotent: safe to re-run.
torch.backends.cudnn.deterministic    = False   # let cuDNN pick aligned kernels
torch.backends.cudnn.benchmark        = True    # autotune for fixed input shape
torch.backends.cudnn.allow_tf32       = True
torch.backends.cuda.matmul.allow_tf32 = True
print('cuDNN preset for DP+AMP:')
print(f'  deterministic = {torch.backends.cudnn.deterministic}')
print(f'  benchmark     = {torch.backends.cudnn.benchmark}')
print(f'  allow_tf32    = {torch.backends.cudnn.allow_tf32}')


## 11b. Smoke Test (1 iter, catches OOM/alignment errors fast)

In [ ]:
# ── Smoke test: 1 forward + backward pass to catch OOM/shape errors fast ─────
import gc

# Hard-fail if stale DataParallel wrapper is present (incompatible with AMP)
assert not isinstance(model, nn.DataParallel), (
    'Model is wrapped in DataParallel. This is incompatible with AMP + '
    'timm ConvNeXt/Swin (misaligned-address bug). Click '
    '"Run -> Restart Kernel & Clear All Outputs" then "Run All" to reset state.'
)

print('Smoke test: 1 train iter + 1 val batch...')
model.train()
imgs, labels, masks = next(iter(train_loader))
imgs   = imgs.to(DEVICE, non_blocking=True)
labels = labels.to(DEVICE, non_blocking=True)
masks  = masks.to(DEVICE, non_blocking=True)

if USE_AMP:
    with autocast():
        cls_out, seg_out, aux_a, aux_b = model(imgs)
        loss, _, _ = multitask_loss(cls_out, seg_out, aux_a, aux_b, labels, masks)
else:
    cls_out, seg_out, aux_a, aux_b = model(imgs)
    loss, _, _ = multitask_loss(cls_out, seg_out, aux_a, aux_b, labels, masks)

print(f'  fwd ok      loss={loss.item():.4f}')
print(f'  cls_out     {tuple(cls_out.shape)}')
print(f'  seg_out     {tuple(seg_out.shape)}')

if USE_AMP:
    scaler.scale(loss).backward()
else:
    loss.backward()
print('  bwd ok')

optimizer.zero_grad(set_to_none=True)

# One val batch
model.train(False)
with torch.no_grad():
    imgs_v, labels_v, masks_v = next(iter(val_loader))
    imgs_v = imgs_v.to(DEVICE, non_blocking=True)
    cls_out, seg_out, _, _ = model(imgs_v)
print(f'  val fwd ok  cls={tuple(cls_out.shape)} seg={tuple(seg_out.shape)}')

# Free smoke test tensors
del imgs, labels, masks, imgs_v, labels_v, masks_v, cls_out, seg_out, aux_a, aux_b, loss
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'  peak VRAM during smoke: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')
    torch.cuda.reset_peak_memory_stats()

print('Smoke test PASSED. Safe to run full training loop below.')


## 12. Training Loop

In [ ]:
history      = []
best_comp    = -float('inf')
patience_ctr = 0
start_time   = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    tr_loss, tr_cls, tr_seg = train_one_epoch(model, train_loader, optimizer, epoch)

    # Free any lingering activation cache between train and val phases
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    val_m = validate(model, val_loader)
    scheduler.step()

    # End-of-epoch cleanup (prevents slow-leak OOMs on long runs)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    row = {
        'epoch': epoch,
        'tr_loss': tr_loss, 'tr_cls': tr_cls, 'tr_seg': tr_seg,
        **{f'val_{k}': v for k, v in val_m.items()},
    }
    history.append(row)

    print(f'[{epoch:02d}/{MAX_EPOCHS}] '
          f'loss={tr_loss:.4f} cls={tr_cls:.4f} seg={tr_seg:.4f} | '
          f'Acc={val_m["acc"]:.4f} AUC={val_m["auc"]:.4f} '
          f'Dice={val_m["dice"]:.4f} FgRec={val_m["fgrec"]:.4f} '
          f'Comp={val_m["comp"]:.4f}')

    if val_m['comp'] > best_comp:
        best_comp = val_m['comp']
        patience_ctr = 0
        # DataParallel was permanently disabled; defensive unwrap kept just
        # in case kernel state still has a wrapped model from a prior session.
        sd = (model.module.state_dict()
              if isinstance(model, nn.DataParallel) else model.state_dict())
        torch.save(sd, BEST_PATH)
        print(f'  -> Best checkpoint saved  (comp={best_comp:.4f})')
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'Early stopping at epoch {epoch}.')
            break

elapsed = time.time() - start_time
print(f'\nTraining complete. Best composite: {best_comp:.4f}  Elapsed: {elapsed/3600:.2f} h')

# Save history
import csv
with open(OUT_DIR / 'history.csv', 'w', newline='') as f:
    if history:
        writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
        writer.writeheader()
        writer.writerows(history)
print(f'history.csv saved.')


## 13. Training Curves

In [ ]:
epochs   = [r['epoch']     for r in history]
tr_loss  = [r['tr_loss']   for r in history]
val_auc  = [r['val_auc']   for r in history]
val_dice = [r['val_dice']  for r in history]
val_comp = [r['val_comp']  for r in history]
val_fgr  = [r['val_fgrec'] for r in history]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f'Training History — {DISPLAY}', fontsize=14, fontweight='bold')

axes[0, 0].plot(epochs, tr_loss, color='steelblue', lw=2)
axes[0, 0].set_title('Train Loss'); axes[0, 0].set_xlabel('Epoch')
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(epochs, val_auc, color='darkorange', lw=2)
axes[0, 1].set_title('Val AUC'); axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylim(0.4, 1.0); axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(epochs, val_dice, color='mediumseagreen', lw=2)
axes[1, 0].set_title('Val Dice'); axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylim(0, 1.0); axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(epochs, val_comp, color='mediumpurple', lw=2, label='Composite')
axes[1, 1].plot(epochs, val_fgr,  color='tomato',       lw=2, linestyle='--', label='FgRecall')
best_ep = epochs[int(np.argmax(val_comp))]
axes[1, 1].axvline(best_ep, color='k', linestyle=':', lw=1.2, label=f'Best ep={best_ep}')
axes[1, 1].set_title('Composite & Forged Recall'); axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylim(0, 1.0); axes[1, 1].legend(fontsize=9); axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'training_curves.png saved.')


## 14. Final Evaluation on Val Set

In [ ]:
# Reload best checkpoint into a fresh model copy (no DataParallel wrapper for eval)
state = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
eval_encoder = timm.create_model(TIMM_NAME, pretrained=False, features_only=True, img_size=512)
eval_channels = eval_encoder.feature_info.channels()
eval_model = FourLevelUNet(eval_encoder, eval_channels,
                            channels_last_input=True).to(DEVICE)
eval_model.load_state_dict(state, strict=False)
eval_model.train(False)
print('Best checkpoint loaded for evaluation.')

# Defensive rebuild: if val_loader was freed or kernel restarted, recreate it
# using the SAME SEED + split logic as the original DataLoaders cell.
try:
    val_loader
except NameError:
    print('val_loader not in scope — rebuilding from val_samples...')
    if 'val_samples' not in globals():
        all_samples, all_uids = build_samples()
        shuffled = sorted(all_uids)
        random.shuffle(shuffled)
        n_val = int(VAL_FRAC * len(shuffled))
        val_uid_set = set(shuffled[:n_val])
        val_samples = [s for s in all_samples if s['uid'] in val_uid_set]
    if 'val_tf' not in globals():
        val_tf = A.Compose([
            A.Resize(height=IMG_SIZE, width=IMG_SIZE),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
    val_ds = ForgeryDataset(val_samples, val_tf)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=0, pin_memory=True,
                            persistent_workers=False)
    print(f'  -> {len(val_loader)} val batches')

# Collect predictions
y_true, y_prob = [], []
seg_probs_all, masks_all = [], []
with torch.no_grad():
    for imgs, labels, masks in tqdm(val_loader, desc='Eval'):
        imgs = imgs.to(DEVICE)
        cls_out, seg_out, _, _ = eval_model(imgs)
        probs     = torch.sigmoid(cls_out).squeeze(1).cpu().numpy()
        seg_probs = torch.sigmoid(seg_out).squeeze(1).cpu().numpy()
        mask_np   = masks.squeeze(1).numpy()
        y_true.extend(labels.numpy())
        y_prob.extend(probs)
        for i in range(len(labels)):
            if labels[i] == 1:
                seg_probs_all.append(seg_probs[i])
                masks_all.append(mask_np[i])

y_true = np.array(y_true)
y_prob = np.array(y_prob)

# Threshold sweep for best F1
best_thr, best_f1 = 0.5, 0.0
for thr in np.arange(0.05, 0.95, 0.01):
    preds = (y_prob > thr).astype(int)
    f1    = f1_score(y_true, preds, zero_division=0)
    if f1 > best_f1:
        best_f1, best_thr = f1, float(thr)

# Dice computed at same threshold
dice_list = []
for sp, gt in zip(seg_probs_all, masks_all):
    pred_bin = (sp > best_thr).astype(np.float32)
    inter    = (pred_bin * gt).sum()
    denom    = pred_bin.sum() + gt.sum() + 1e-7
    dice_list.append(2 * inter / denom)

y_pred = (y_prob > best_thr).astype(int)
cm     = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
auc  = roc_auc_score(y_true, y_prob)
dice = float(np.mean(dice_list)) if dice_list else float('nan')
acc  = accuracy_score(y_true, y_pred)
prec = tp / (tp + fp + 1e-7)
rec  = tp / (tp + fn + 1e-7)
spec = tn / (tn + fp + 1e-7)
f1v  = 2 * tp / (2 * tp + fp + fn + 1e-7)

metrics = {
    'model_name': MODEL_NAME,
    'display':    DISPLAY,
    'threshold':  float(best_thr),
    'accuracy':   float(acc),
    'auc':        float(auc),
    'precision':  float(prec),
    'forged_recall': float(rec),
    'specificity': float(spec),
    'f1':         float(f1v),
    'mean_dice':  float(dice),
    'composite':  float(0.4 * auc + 0.3 * rec + 0.3 * dice),
    'cm_tn':      int(tn), 'cm_fp': int(fp), 'cm_fn': int(fn), 'cm_tp': int(tp),
}

with open(OUT_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))


## 15. Dashboard

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Validation — {DISPLAY}', fontsize=14, fontweight='bold')

ax = axes[0]
ax.hist(y_prob[y_true == 0], bins=40, alpha=0.65, color='steelblue',
        label=f'Authentic n={(y_true==0).sum()}')
ax.hist(y_prob[y_true == 1], bins=40, alpha=0.65, color='tomato',
        label=f'Forged    n={(y_true==1).sum()}')
ax.axvline(best_thr, color='k', linestyle='--', lw=1.5,
           label=f'thr={best_thr:.2f}')
ax.set_xlabel('Predicted probability'); ax.set_ylabel('Count')
ax.set_title('Score Distribution'); ax.legend(fontsize=9)

ax = axes[1]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Authentic', 'Forged'],
            yticklabels=['Authentic', 'Forged'],
            annot_kws={'size': 16})
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix (thr={best_thr:.2f})')

ax = axes[2]
fpr, tpr, _ = roc_curve(y_true, y_prob)
ax.plot(fpr, tpr, lw=2.5, color='darkorange', label=f'AUC={auc:.4f}')
ax.fill_between(fpr, tpr, alpha=0.10, color='darkorange')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curve'); ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig(OUT_DIR / 'dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print('dashboard.png saved.')

print('=' * 50)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'  {k:18s}: {v:.4f}')
    else:
        print(f'  {k:18s}: {v}')
print('=' * 50)
print()
print(classification_report(y_true, y_pred, target_names=['Authentic', 'Forged']))


## 16. Qualitative Predictions Grid

In [ ]:
# Forged val examples grid: original | overlay | heatmap | GT mask
def postprocess_mask(prob, threshold=0.5, min_area=200):
    binary = (prob > threshold).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    labeled, n = scipy_label(binary)
    out = np.zeros_like(binary)
    for i in range(1, n + 1):
        if (labeled == i).sum() >= min_area:
            out[labeled == i] = 1
    return out.astype(np.float32)


N_SHOW = 12
forged_samples = [s for s in val_samples if s['label'] == 1]
show = random.sample(forged_samples, min(N_SHOW, len(forged_samples)))

if not show:
    print('No forged val samples.')
else:
    fig, axes = plt.subplots(len(show), 4, figsize=(20, 5 * len(show)))
    if len(show) == 1:
        axes = axes[np.newaxis, :]
    titles = ['Original', 'Overlay (red=pred, green=GT)', 'Heatmap', 'GT mask']
    for c, t in enumerate(titles):
        axes[0, c].set_title(t, fontsize=10, fontweight='bold')

    test_tf = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

    for r, s in enumerate(tqdm(show, desc='Render preds')):
        rgb = cv2.cvtColor(cv2.imread(s['img']), cv2.COLOR_BGR2RGB)
        h, w = rgb.shape[:2]
        gt = load_mask(s['mask'], h, w) if s['mask'] else np.zeros((h, w), np.float32)

        ten = test_tf(image=rgb)['image'].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            cls_o, seg_o, _, _ = eval_model(ten)
        cp = torch.sigmoid(cls_o).item()
        sp = torch.sigmoid(seg_o[0, 0]).cpu().numpy()
        sp_up = cv2.resize(sp, (w, h), interpolation=cv2.INTER_LINEAR)
        bin_mask = postprocess_mask(sp_up, threshold=best_thr, min_area=200)

        ov = rgb.copy().astype(np.float32)
        ov[bin_mask > 0] = ov[bin_mask > 0] * 0.45 + np.array([255, 40, 40]) * 0.55
        ov = np.clip(ov, 0, 255).astype(np.uint8)
        contours, _ = cv2.findContours(gt.astype(np.uint8),
                                        cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(ov, contours, -1, (0, 220, 0), 2)

        heat = cv2.cvtColor(
            cv2.applyColorMap((sp_up * 255).astype(np.uint8), cv2.COLORMAP_JET),
            cv2.COLOR_BGR2RGB)

        gt_vis = (gt * 255).astype(np.uint8)
        d = 2 * (bin_mask * gt).sum() / (bin_mask.sum() + gt.sum() + 1e-7)

        axes[r, 0].imshow(rgb)
        axes[r, 0].text(-0.02, 0.5,
                        f'UID:{s["uid"]}\np={cp:.3f}  D={d:.3f}',
                        transform=axes[r, 0].transAxes,
                        fontsize=7, va='center', ha='right',
                        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
        axes[r, 1].imshow(ov)
        axes[r, 2].imshow(heat)
        axes[r, 3].imshow(gt_vis, cmap='gray', vmin=0, vmax=255)
        for ax in axes[r]:
            ax.axis('off')

    plt.suptitle(f'Forged predictions — {DISPLAY}', fontsize=15, fontweight='bold', y=1.001)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'predictions.png', dpi=120, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('predictions.png saved.')


## 17. Save Final Model Bundle

In [ ]:
# Save final model bundle (for the webapp).
# eval_model holds the best checkpoint reloaded fresh; using it directly
# avoids dependency on `model` (which may have been freed after training).
final_state = eval_model.state_dict()

torch.save({
    'model_state_dict': final_state,
    'cls_threshold':    float(best_thr),
    'seg_threshold':    float(best_thr),
    'img_size':         IMG_SIZE,
    'backbone':         TIMM_NAME,
    'architecture':     DISPLAY,
    'val_auc':          float(auc),
    'val_dice':         float(dice),
    'val_composite':    float(metrics['composite']),
    'val_specificity':  float(spec),
    'val_forged_recall': float(rec),
}, FINAL_PATH)
sz = os.path.getsize(FINAL_PATH) / 1e6
print(f'Saved {FINAL_PATH}  ({sz:.1f} MB)')
print(f'  AUC : {auc:.4f}')
print(f'  Dice: {dice:.4f}')
print(f'  Spec: {spec:.4f}  <-- key metric vs baseline')


## 18. Memory Cleanup (mandatory)

In [ ]:
# ── Memory cleanup (research-grade hygiene) ──────────────────────────────────
print(f'Peak VRAM during run: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

del model, optimizer, scaler, train_loader, val_loader
try:
    del eval_model, eval_encoder
except NameError:
    pass
try:
    del encoder
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print('Memory released. Restart kernel before running the next training notebook.')
print(f'Outputs at: {OUT_DIR}')
for p in sorted(OUT_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size/1024:.1f} KB)')
